In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json


In [2]:
import sys
from pathlib import Path

# Replace with your GitHub Repository URL
GITHUB_REPO_URL = "https://github.com/arpitkumar2004/BeyondCitation.git"
REPO_NAME = "BeyondCitation"

# Clone repo or pull latest updates
if not Path(REPO_NAME).exists():
    print(f"Cloning GitHub Repository from: {GITHUB_REPO_URL}...")
    !git clone {GITHUB_REPO_URL}
else:
    print(f"Pulling latest updates from GitHub in {REPO_NAME}...")
    !cd {REPO_NAME} && git pull

# Add repository root to python module path
repo_path = Path.cwd() / REPO_NAME if Path(REPO_NAME).exists() else Path.cwd()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Install package dependencies
print("Installing package dependencies...")
!pip install -q -e ./{REPO_NAME} optuna polars duckdb faiss-cpu sentence-transformers


Cloning GitHub Repository from: https://github.com/arpitkumar2004/BeyondCitation.git...
Cloning into 'BeyondCitation'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 151 (delta 30), reused 149 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 11.23 MiB | 26.57 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Installing package dependencies...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.3 MB/s eta 0:00:0

In [3]:
import torch

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Hardware] PyTorch execution device: {device.upper()}")
if device == "cuda":
    print(f"[GPU Name] {torch.cuda.get_device_name(0)}")

# Candidate Kaggle dataset input paths
candidate_paths = [
    "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json",
    "/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json",
    "data/raw/arxiv-metadata-oai-snapshot.json"
]

dataset_path = None
for cp in candidate_paths:
    if Path(cp).exists():
        dataset_path = cp
        break

if dataset_path:
    print(f"[Dataset] Found arXiv snapshot at: {dataset_path}")
else:
    print("[Dataset] No arXiv JSON file found in /kaggle/input. Execution will use synthetic dry-run dataset.")


[Hardware] PyTorch execution device: CUDA
[GPU Name] Tesla T4
[Dataset] Found arXiv snapshot at: /kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json


In [4]:
from main import run_pipeline, parse_arguments

# Configure execution parameters
args_list = [
    "--output-dir", "/kaggle/working/outputs",
    "--sample-size", "10000",
    "--enable-optuna",
    "--n-trials", "30"
]

if dataset_path:
    args_list.extend(["--input-path", dataset_path])
else:
    args_list.append("--dry-run")

print(f"Launching pipeline with arguments: {' '.join(args_list)}")
sys.argv = ["main.py"] + args_list
args = parse_arguments()

# Execute Full 13-Arm Benchmark + Optuna Pareto Optimization
run_pipeline(args)


Launching pipeline with arguments: --output-dir /kaggle/working/outputs --sample-size 10000 --enable-optuna --n-trials 30 --input-path /kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json
BEYOND CITATION: HIGH-PERFORMANCE MULTI-OBJECTIVE PAPER RECOMMENDER PIPELINE
Ingesting arXiv snapshot from: /kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json


Fast Ingestion (arXiv Snapshot): 100%|██████████| 100000/100000 [00:03<00:00, 32367.39it/s]


Loaded 58 target papers (`cs.CL`, `cs.LG`, `cs.AI`, `cs.IR`).
Ingested 58 candidate papers.

--- Constructing Representation Spaces & FAISS Vector Indices ---
[M1] Encoding 58 documents using 'TF-IDF Baseline'...
[M1] Embeddings cached to '/kaggle/working/outputs/embeddings/M1_58_emb.npy' (Shape: (58, 512)).
[M2] Encoding 58 documents using 'Doc2Vec / Dense Unsupervised'...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[M7] Embeddings cached to '/kaggle/working/outputs/embeddings/M7_58_emb.npy' (Shape: (58, 768)).
Initialized 6 representation spaces (TF-IDF, SBERT, Doc2Vec, SciBERT, SPECTER2, LinkBERT) with FAISS retrieval.

--- Optuna Multi-Objective Pareto Weight Tuning ---
Running Optuna NSGA-II Multi-Objective Optimization (30 trials)...
Optuna complete! Found 15 Pareto-optimal configurations.
Top Optuna Pareto Configuration: {'trial_number': 1, 'w_sem': 1.8212348635689741, 'w_link': 1.0626028889192785, 'w_rec': 0.25863560116787565, 'w_pop': 0.303509276340179, 'ndcg_5': 0.2717378194368253, 'novelty_5': 0.8560000000000001}

--- Executing Full 13-Run Benchmark Evaluation ---


Benchmark Arm Evaluation: 100%|██████████| 50/50 [00:01<00:00, 47.10it/s]



S-TIER JOURNAL BENCHMARK COMPARISON TABLE (13 EXPERIMENTAL RUNS WITH STATISTICAL SIGNIFICANCE)
Significance markers vs. LinkBERT: *** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant
Exp ID        Experimental Model / Run      NDCG@5 (High)      MRR (High) Novelty@5 (High)  Diversity ILD (High)  Gini Popularity Bias (Low) p-value (vs LinkBERT)
    M1                          TF-IDF  0.0967 ± 0.0355ns 0.2030 ± 0.0873  0.3840 ± 0.0843                0.8847                      0.5425                0.0852
    M2                         Doc2Vec  0.1147 ± 0.0396ns 0.2350 ± 0.0904  0.3640 ± 0.0776                0.5240                      0.5256                0.7035
    M3              SBERT (all-MiniLM)   0.0864 ± 0.0340* 0.1970 ± 0.0888  0.3840 ± 0.0858                0.5818                      0.5330                0.0217
    M4                         SciBERT  0.1006 ± 0.0379ns 0.2200 ± 0.0908  0.3800 ± 0.0772                0.1022                      0.5339                